In [ ]:
import pandas as pd
import cv2
import numpy as np
import re
import subprocess

from pathlib import Path
from tqdm import tqdm
from datetime import timedelta, datetime

project_dir = Path('..').resolve()

video_dir = Path('/Volumes/LAB-DRIVE/telfer-thesis/raw-unsorted/undone/')

In [ ]:
list(video_dir.glob('Basler*__*.mp4'))

## Find Video File from Meta Details

In [ ]:
df = pd.read_csv(project_dir / 'data/raw/raw_videos.csv')

def get_original_filename(row):
    return f""

def match_to_file(row):
    s = f'{row.camera}__*__{row.year:04}{row.month:02}{row.day:02}_{row.hour:02}{row.minutes:02}{row.seconds:02}*.mp4'
    try:
        return next(video_dir.glob(s)).parts[-1]
    except:
        return ""
    
df['original_video'] = df.apply(match_to_file, axis=1)

In [ ]:
df = df[df.original_video != ''].copy()
df

In [ ]:
temp = df.dropna(subset=['animal', 'recording'])
groups = temp.groupby(by=['animal', 'recording'])
for (animal, recording), group_df in groups:
    if len(group_df) > 1:
        print(animal, recording, len(group_df))

## Crop Video

In [ ]:

def to_seconds(time_str):
    d = datetime.strptime(time_str,"%M:%S")
    d = timedelta(minutes=d.minute, seconds=d.second)
    return int(d.total_seconds())

def seconds_to_str(seconds):
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{minutes:02}:{seconds:02}"

def preprocess_video(row):
    if row.discard == 1:
        return 
    
    cmd = "ffmpeg -hide_banner -loglevel error"

    # Add a start flag if start time is specified
    if row.start != '0' and row.start != 'nan':
        cmd += f' -ss {row.start}'
    
    # Add a duration flag if end time is specified
    if row.end != '-1' and row.end != 'nan':
        if row.start != '0' and row.start != 'nan':
            duration = seconds_to_str(to_seconds(row.end) - to_seconds(row.start))
        else:
            duration = row.end

        cmd += f' -t {duration}'

    # Add the video in
    cmd += f' -i "{str(video_dir / row.original_video)}"'

    # Specify codec
    cmd += ' -c copy'

    # Output file
    recording_labels = dict(enumerate(['acclimation', 'preinjection', '1h-postinjection', '2h-postinjection', '4h-postinjection']))
    fname = f"{row.animal}_{recording_labels[int(row.recording)]}.mp4"
    cmd += f' "{output_dir / fname}"'

    return cmd

output_dir = Path('/Volumes/LAB-DRIVE/telfer-thesis/output-test')
! rm -rf {output_dir}
! mkdir -p {output_dir}

df['start'] = df['start'].astype(str)
df['end'] = df['end'].astype(str)

for idx, row in df.iterrows():
    cmd = preprocess_video(row)

    if cmd: 
        print(cmd)
        process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE)
        process.wait()
    

In [ ]:
import subprocess
process = subprocess.Popen("ffmpeg -hide_banner -loglevel error", shell=True, stdout=subprocess.PIPE)
process.wait()